# OSimFlow Campaign Analysis with DuckDB

This notebook demonstrates how to query OSimFlow campaign results stored in Parquet format
using [DuckDB](https://duckdb.org/) — an in-process analytical database that runs SQL
directly on Parquet files without loading them into memory first.

### OSimFlow output structure

After a campaign completes, the output directory (`outdir/`) contains:

| File | Description |
|------|-------------|
| `aggregated_results.parquet` | Per-sample KPIs: EUI, total energy, heating/cooling breakdown, status |
| `failed_simulations.parquet` | Samples that failed with error summaries from `eplusout.err` |
| `timeseries_aggregated.parquet` | Monthly/daily aggregated time-series by variable |
| `run.json` | Campaign execution trace (step timing, cache stats) |

> **Note:** Column names follow BEM conventions. Adjust if your campaign uses custom KPI extractors.

## Setup — imports and configuration

Import DuckDB and pandas. DuckDB can query Parquet files directly via `read_parquet()`.
We also import matplotlib for visualisations.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# ── Point this at your campaign output directory ──────────────────────
OUTDIR = "../results"

# Primary KPI table (Parquet preferred, falls back to CSV)
import os
agg_path = os.path.join(OUTDIR, "aggregated_results.parquet")
if not os.path.exists(agg_path):
    agg_path = os.path.join(OUTDIR, "aggregated_results.csv")

failed_path = os.path.join(OUTDIR, "failed_simulations.parquet")
if not os.path.exists(failed_path):
    failed_path = os.path.join(OUTDIR, "failed_simulations.csv")

print(f"KPI file:  {agg_path}")
print(f"Failed file: {failed_path}")

## Connect and preview the data

Open an in-memory DuckDB connection and load the aggregated results.
The table is registered as `agg` for subsequent queries.

In [ ]:
con = duckdb.connect()

# Register the file as a queriable table
# DuckDB auto-detects Parquet vs CSV from the extension.
con.execute(f"CREATE OR REPLACE VIEW agg AS SELECT * FROM read_parquet('{agg_path}')")

# Quick preview
con.execute("DESCRIBE agg").df()

## Monthly energy consumption by end-use

Summarise total energy broken down by heating, cooling, lighting, and other end-uses.
This requires the `timeseries_aggregated` file, which contains per-month values.
If only annual KPIs are available, the query falls back to the aggregate table.

In [ ]:
# Column names below are BEM defaults.
# Adjust to match your campaign's actual schema (check DESCRIBE agg above).
monthly_energy = con.execute("""
    SELECT
        sample_id,
        heating_kwh   AS heating,
        cooling_kwh   AS cooling,
        lighting_kwh  AS lighting,
        equipment_kwh AS equipment,
        fans_pumps_kwh AS fans_pumps,
        total_energy_kwh AS total
    FROM agg
    WHERE status = 'SUCCESS'
    ORDER BY sample_id
""").df()

monthly_energy.head(10)

## Peak demand profiling

Identify the samples with the highest peak demand. This is useful for
electrical sizing studies and utility tariff analysis.

If the campaign outputs a `peak_demand_kw` column, we use it directly;
otherwise we estimate from total energy (rough proxy).

In [ ]:
# Peak demand analysis — adjust column names to your schema
peak_demand = con.execute("""
    SELECT
        sample_id,
        peak_demand_kw,
        total_energy_kwh,
        ROUND(total_energy_kwh / 8760.0, 2) AS avg_demand_kw,
        ROUND(peak_demand_kw / NULLIF(total_energy_kwh / 8760.0, 0), 2) AS load_factor
    FROM agg
    WHERE status = 'SUCCESS'
      AND peak_demand_kw IS NOT NULL
    ORDER BY peak_demand_kw DESC
    LIMIT 20
""").df()

peak_demand

## EUI distribution analysis

Energy Use Intensity (EUI, in kWh/m²/yr) is the headline metric for building
energy performance. This cell computes summary statistics and plots a histogram
to show the distribution across all successful samples.

In [ ]:
eui = con.execute("""
    SELECT
        eui_kwh_m2_yr
    FROM agg
    WHERE status = 'SUCCESS'
      AND eui_kwh_m2_yr IS NOT NULL
    ORDER BY eui_kwh_m2_yr
""").df()

stats = con.execute("""
    SELECT
        COUNT(*)                AS n_samples,
        ROUND(AVG(eui_kwh_m2_yr), 1)   AS mean_eui,
        ROUND(MEDIAN(eui_kwh_m2_yr), 1) AS median_eui,
        ROUND(MIN(eui_kwh_m2_yr), 1)    AS min_eui,
        ROUND(MAX(eui_kwh_m2_yr), 1)    AS max_eui,
        ROUND(STDDEV(eui_kwh_m2_yr), 1) AS std_eui
    FROM agg
    WHERE status = 'SUCCESS'
      AND eui_kwh_m2_yr IS NOT NULL
""").df()

print(stats.to_string(index=False))

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eui["eui_kwh_m2_yr"], bins=30, edgecolor="black", alpha=0.75)
ax.set_xlabel("EUI (kWh/m²/yr)")
ax.set_ylabel("Number of samples")
ax.set_title("EUI Distribution Across Campaign Samples")
ax.axvline(eui["eui_kwh_m2_yr"].median(), color="red", linestyle="--", label="Median")
ax.legend()
plt.tight_layout()
plt.show()

## Failed simulations summary

Quick overview of which samples failed and why. The `failed_simulations` table
contains the first severe error line from each `eplusout.err`.

In [ ]:
# Read the failed simulations file (Parquet or CSV)
con.execute(f"CREATE OR REPLACE VIEW failed AS SELECT * FROM read_parquet('{failed_path}')")

failed_summary = con.execute("""
    SELECT
        sample_id,
        status,
        error_summary
    FROM failed
    ORDER BY sample_id
""").df()

print(f"Total failed samples: {len(failed_summary)}")
failed_summary

## Cross-sample comparison — top performers by EUI

Rank samples by EUI to identify the best-performing parametric variants.
This is the typical starting point for design-optimization studies.

In [ ]:
top_performers = con.execute("""
    SELECT
        sample_id,
        ROUND(eui_kwh_m2_yr, 1)  AS eui,
        ROUND(total_energy_kwh, 0) AS total_kwh,
        ROUND(heating_kwh, 0)      AS heating_kwh,
        ROUND(cooling_kwh, 0)      AS cooling_kwh,
        status
    FROM agg
    WHERE status = 'SUCCESS'
      AND eui_kwh_m2_yr IS NOT NULL
    ORDER BY eui_kwh_m2_yr ASC
    LIMIT 5
""").df()

print("Top 5 samples by EUI (lowest = best):")
top_performers

## Cleanup

Close the DuckDB connection when done.

In [ ]:
con.close()
print("Done.")